In [1]:
import chromadb
from models import Document,DocumentSection, Chunk,RetrievedChunk,Query
from pprint import pprint
chroma_client = chromadb.Client()


In [2]:
import csv

with open('documents.csv') as file:
    lines = csv.reader(file)

    documents = []
    metadatas = []
    ids = []

    for i, line in enumerate(lines):
        if i == 0:
            continue  # skip header row

        documents.append(line[1])                    # document_title
        metadatas.append({"document_id": line[0]})  # metadata
        ids.append(line[0])                          # document_id

In [3]:
collection = chroma_client.create_collection(name="my_collection")

In [4]:
sample_documents = [
    Document(
        document_id="KB-001",
        source_type="kb",
        source_ref="KB-001",
        title="Reset VPN Access",
        metadata={"product": "vpn", "audience": "staff"},
        sections=[
            DocumentSection("title", "Reset VPN Access"),
            DocumentSection("description", "User cannot connect after a password change."),
            DocumentSection("solution", "Remove the saved VPN profile, sign in again, and re-approve MFA."),
            DocumentSection("notes", "Applies to managed Windows laptops."),
        ],
    ),
    Document(
        document_id="KB-002",
        source_type="kb",
        source_ref="KB-002",
        title="Install Office 365",
        metadata={"product": "office", "audience": "staff"},
        sections=[
            DocumentSection("title", "Install Office 365"),
            DocumentSection("description", "Guide for installing Office 365 applications on a new device."),
            DocumentSection("solution", "Open Company Portal, search for Microsoft 365 Apps, and click Install."),
            DocumentSection("notes", "Requires an active Microsoft 365 license assigned in Azure AD."),
        ],
    ),
    Document(
        document_id="KB-003",
        source_type="kb",
        source_ref="KB-003",
        title="Unlock Active Directory Account",
        metadata={"product": "active_directory", "audience": "helpdesk"},
        sections=[
            DocumentSection("title", "Unlock Active Directory Account"),
            DocumentSection("description", "User account locked out after too many failed login attempts."),
            DocumentSection("solution", "Open ADUC, locate the user, right-click and select Unlock Account. Reset password if required."),
            DocumentSection("notes", "Lockout threshold is 5 attempts. Check for credential stuffing if repeated."),
        ],
    ),
    Document(
        document_id="KB-004",
        source_type="kb",
        source_ref="KB-004",
        title="Configure Email Signature in Outlook",
        metadata={"product": "outlook", "audience": "staff"},
        sections=[
            DocumentSection("title", "Configure Email Signature in Outlook"),
            DocumentSection("description", "Staff need to set up a standardised email signature."),
            DocumentSection("solution", "Go to File > Options > Mail > Signatures. Create a new signature using the approved company template."),
            DocumentSection("notes", "Template available on the intranet under Brand Guidelines."),
        ],
    ),
    Document(
        document_id="KB-005",
        source_type="kb",
        source_ref="KB-005",
        title="Map a Network Drive",
        metadata={"product": "windows", "audience": "staff"},
        sections=[
            DocumentSection("title", "Map a Network Drive"),
            DocumentSection("description", "User cannot access shared file storage from their workstation."),
            DocumentSection("solution", "Open File Explorer, click This PC, then Map Network Drive. Enter the UNC path provided by IT."),
            DocumentSection("notes", "Must be connected to VPN if working remotely."),
        ],
    ),
    Document(
        document_id="KB-006",
        source_type="kb",
        source_ref="KB-006",
        title="Enrol Device in Intune MDM",
        metadata={"product": "intune", "audience": "helpdesk"},
        sections=[
            DocumentSection("title", "Enrol Device in Intune MDM"),
            DocumentSection("description", "New or wiped device needs to be enrolled in mobile device management."),
            DocumentSection("solution", "Sign in to Windows with corporate credentials. Intune enrolment triggers automatically via Autopilot."),
            DocumentSection("notes", "Device must have internet access during OOBE. Contact helpdesk if enrolment fails at 40%."),
        ],
    ),
    Document(
        document_id="KB-007",
        source_type="kb",
        source_ref="KB-007",
        title="Reset MFA Authenticator App",
        metadata={"product": "mfa", "audience": "helpdesk"},
        sections=[
            DocumentSection("title", "Reset MFA Authenticator App"),
            DocumentSection("description", "User has lost or replaced their phone and cannot complete MFA."),
            DocumentSection("solution", "Admin navigates to Azure AD > Users > Authentication Methods and deletes the existing MFA registration. User re-registers on next login."),
            DocumentSection("notes", "Verify user identity via video call before resetting. Log the action in the ticketing system."),
        ],
    ),
    Document(
        document_id="KB-008",
        source_type="kb",
        source_ref="KB-008",
        title="Printer Not Found on Network",
        metadata={"product": "printing", "audience": "staff"},
        sections=[
            DocumentSection("title", "Printer Not Found on Network"),
            DocumentSection("description", "User cannot see or connect to a shared network printer."),
            DocumentSection("solution", "Open Settings > Printers & Scanners > Add a Printer. If not listed, enter the printer IP manually or run the Add Printer wizard with the UNC path."),
            DocumentSection("notes", "Printer drivers are pushed via Group Policy. Restart print spooler if driver install fails."),
        ],
    ),
    Document(
        document_id="KB-009",
        source_type="kb",
        source_ref="KB-009",
        title="Request Software Installation",
        metadata={"product": "software", "audience": "staff"},
        sections=[
            DocumentSection("title", "Request Software Installation"),
            DocumentSection("description", "Staff need software not available in the Company Portal."),
            DocumentSection("solution", "Submit a request via the IT service portal with business justification. Helpdesk will assess and deploy via Intune if approved."),
            DocumentSection("notes", "Unapproved software must not be installed by end users. Violations are logged by endpoint monitoring."),
        ],
    ),
    Document(
        document_id="KB-010",
        source_type="kb",
        source_ref="KB-010",
        title="Recover Deleted Email in Outlook",
        metadata={"product": "outlook", "audience": "staff"},
        sections=[
            DocumentSection("title", "Recover Deleted Email in Outlook"),
            DocumentSection("description", "User accidentally deleted an important email and emptied the trash."),
            DocumentSection("solution", "In Outlook, go to Folder > Recover Deleted Items. Select the email and click Restore. Items are recoverable for 30 days."),
            DocumentSection("notes", "For items older than 30 days, a compliance eDiscovery request must be raised by a manager."),
        ],
    ),
    Document(
        document_id="KB-011",
        source_type="kb",
        source_ref="KB-011",
        title="Set Up Out of Office Reply",
        metadata={"product": "outlook", "audience": "staff"},
        sections=[
            DocumentSection("title", "Set Up Out of Office Reply"),
            DocumentSection("description", "User wants to configure an automatic reply while on leave."),
            DocumentSection("solution", "In Outlook go to File > Automatic Replies. Set the date range and compose separate messages for internal and external senders."),
            DocumentSection("notes", "Avoid including sensitive information in external auto-replies."),
        ],
    ),
    Document(
        document_id="KB-012",
        source_type="kb",
        source_ref="KB-012",
        title="Fix Slow Laptop Performance",
        metadata={"product": "windows", "audience": "staff"},
        sections=[
            DocumentSection("title", "Fix Slow Laptop Performance"),
            DocumentSection("description", "User reports laptop is unusually slow, especially at startup."),
            DocumentSection("solution", "Check Task Manager for high CPU or memory processes. Disable unnecessary startup programs. Run Windows Update. If persistent, escalate for hardware review."),
            DocumentSection("notes", "Devices over 4 years old with less than 8GB RAM are flagged for refresh."),
        ],
    ),
]

In [ ]:
collection.add(
    documents = [section.content for doc in sample_documents for section in doc.sections],
    metadatas=[doc.metadata for doc in sample_documents for _ in doc.sections],
    ids=[f"{doc.document_id}_{i}" for doc in sample_documents for i, section in enumerate(doc.sections)],
)

In [13]:
results = collection.query(
    query_texts=["how do i fix my laptop"],
    n_results=5,   
    include=["documents", "metadatas",] 
)
pprint(results)

{'data': None,
 'distances': None,
 'documents': [['Fix Slow Laptop Performance',
                'User reports laptop is unusually slow, especially at startup.',
                'Applies to managed Windows laptops.',
                'Check Task Manager for high CPU or memory processes. Disable '
                'unnecessary startup programs. Run Windows Update. If '
                'persistent, escalate for hardware review.',
                'Submit a request via the IT service portal with business '
                'justification. Helpdesk will assess and deploy via Intune if '
                'approved.']],
 'embeddings': None,
 'ids': [['KB-012_0', 'KB-012_1', 'KB-001_3', 'KB-012_2', 'KB-009_2']],
 'included': ['documents', 'metadatas'],
 'metadatas': [[{'audience': 'staff', 'product': 'windows'},
                {'audience': 'staff', 'product': 'windows'},
                {'audience': 'staff', 'product': 'vpn'},
                {'audience': 'staff', 'product': 'windows'},
        

In [9]:
chroma_client.delete_collection(name="my_collection")